# Week 9 Coding Practice: Relational Data in Applied Data Science

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/DataScience-book/blob/w09-cp-01-colab-link/module09/week09_relational_data_practice.ipynb)

**Estimated time:** 60–90 minutes
**Format:** ungraded guided practice

This activity uses a small bike-share dataset with two related tables: `stations` and `trips`. The goal is not to memorize SQL syntax. Instead, the focus is on turning related records into a trustworthy analysis dataset and then using that dataset to answer a practical question.

The main question is:

**How does recorded departure activity vary across stations and neighborhoods during a defined observation period?**

Recorded departures describe observed use. They do not, by themselves, establish unmet demand or justify adding or removing stations.


## Learning objectives

By the end of this activity, you should be able to:

1. explain the relationship between a station table and a trip table;
2. inspect raw relational data for missing values and invalid references;
3. compare different join strategies and explain what each type of join keeps;
4. diagnose a misleading count after a left join and repair it;
5. build a station-level analysis table with one row per station;
6. validate the final table before interpreting it; and
7. use the validated station table to answer a new neighborhood-level question.


## 1. Load and inspect the data

We will work with a synthetic bike-share extract containing:

- `stations`: one row per station, with neighborhood information;
- `trips`: one row per recorded trip, including a start-station reference and a trip duration when available.

The raw trip table contains some integrity problems by design, including invalid or missing start-station IDs and missing duration values. This is realistic for a raw extract and is exactly why we validate before interpreting.


In [ ]:
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')

candidate_paths = [
    Path('appliedDataScience/module09/stations.csv'),
    Path('module09/stations.csv'),
    Path.cwd() / 'appliedDataScience/module09/stations.csv',
]
stations_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])

trip_paths = [
    Path('appliedDataScience/module09/trips.csv'),
    Path('module09/trips.csv'),
    Path.cwd() / 'appliedDataScience/module09/trips.csv',
]
trips_path = next((p for p in trip_paths if p.exists()), trip_paths[0])

print('Stations path:', stations_path)
print('Trips path:', trips_path)

stations = pd.read_csv(stations_path)
trips = pd.read_csv(trips_path)

conn = sqlite3.connect(':memory:')
stations.to_sql('stations', conn, index=False, if_exists='replace')
trips.to_sql('trips', conn, index=False, if_exists='replace')

print('Stations rows:', len(stations))
print('Trips rows:', len(trips))
print(stations.head())
print(trips.head())


### Reasoning checkpoint

Before running the SQL below, answer these questions in writing:

1. What does one row in `stations` represent?
2. What does one row in `trips` represent?
3. Which field links trips to stations?
4. Should a station that has no recorded departures still appear in the final station-level table?
5. What should one row represent in the final analysis dataset?


## 2. Inspect the raw tables

Start by checking the size of each table and the relationship between keys. The first query is intentionally short and should help you decide whether the data look consistent enough to proceed.


In [ ]:
query = """
SELECT
    'stations' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT station_id) AS distinct_station_ids
FROM stations

UNION ALL

SELECT
    'trips' AS table_name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_id) AS distinct_trip_ids
FROM trips;
"""

pd.read_sql_query(query, conn)


In [ ]:
# TODO: Modify the query to inspect missing values and invalid station references.
# Use the trips table and check:
# - the number of rows;
# - the number of times start_station_id is NULL;
# - the number of trips with a start_station_id not found in stations;
# - a few example rows with missing or invalid station IDs.

query = """
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN start_station_id IS NULL THEN 1 ELSE 0 END) AS missing_start_station_id,
    SUM(CASE WHEN start_station_id IS NOT NULL AND start_station_id NOT IN (SELECT station_id FROM stations) THEN 1 ELSE 0 END) AS unmatched_start_station_id
FROM trips;
"""

pd.read_sql_query(query, conn)


### Explain what the checks establish

- A `NULL` station ID means the trip record does not identify a start station.
- A station ID that does not appear in `stations` is an unmatched reference.
- These are not the same concept. A trip can have a missing key or an invalid key, and both cases matter when we create a trustworthy analytical dataset.


## 3. Predict and compare joins

Before running the next queries, write down the answers to these questions:

- Which stations will disappear from an `INNER JOIN` between `stations` and `trips`?
- Which rows will remain in a `LEFT JOIN` from `stations` to `trips`?
- Will station identifiers repeat after joining to trips? Why?
- If a trip references a non-existent station ID, will it be visible in a left join from `stations`?


In [ ]:
inner_join = """
SELECT s.station_id, s.station_name, s.neighborhood, t.trip_id, t.duration_minutes
FROM stations s
INNER JOIN trips t
    ON s.station_id = t.start_station_id
ORDER BY s.station_id, t.trip_id
LIMIT 12;
"""

pd.read_sql_query(inner_join, conn)


In [ ]:
left_join = """
SELECT s.station_id, s.station_name, s.neighborhood, t.trip_id, t.duration_minutes
FROM stations s
LEFT JOIN trips t
    ON s.station_id = t.start_station_id
ORDER BY s.station_id, t.trip_id
LIMIT 12;
"""

pd.read_sql_query(left_join, conn)


### Interpretation

The `INNER JOIN` returns only trips with a valid station match. The `LEFT JOIN` preserves all stations, including those with no recorded departures. But notice the important limitation: a left join from `stations` does not reveal every trip with an invalid or missing start-station reference. Those trips can only be found by checking the `trips` table directly.


## 4. Diagnose a misleading summary

The next SQL query looks reasonable, but it is intentionally misleading. It uses `COUNT(*)` after a left join, and it counts one row for a station with no trips even though the station had zero departures.


In [ ]:
misleading_query = """
SELECT
    s.station_id,
    s.station_name,
    COUNT(*) AS reported_departures
FROM stations s
LEFT JOIN trips t
    ON s.station_id = t.start_station_id
GROUP BY s.station_id, s.station_name
ORDER BY s.station_id;
"""

pd.read_sql_query(misleading_query, conn)


### Questions to answer

1. Why does the query run successfully even when a station has no departures?
2. What is `COUNT(*)` actually counting in this joined table?
3. How should the query be corrected?
4. Why is it wrong to treat `COUNT(*)` as a valid departure count after a left join?
5. How does this differ from `COUNT(duration_minutes)`?


In [ ]:
# TODO: Rewrite the joined summary so it counts trip_id values instead of joined rows.
# After fixing it, verify the result for a station with no departures.

query = """
SELECT
    s.station_id,
    s.station_name,
    COUNT(t.trip_id) AS recorded_departures,
    COUNT(t.duration_minutes) AS departures_with_observed_duration
FROM stations s
LEFT JOIN trips t
    ON s.station_id = t.start_station_id
GROUP BY s.station_id, s.station_name
ORDER BY s.station_id;
"""

pd.read_sql_query(query, conn)


## 5. Build an analysis-ready station table

The final table should contain one row per station and at least these columns:

- station_id
- station_name
- neighborhood
- recorded_departure_count
- departures_with_observed_duration
- mean_duration_minutes

A good strategy is:

1. aggregate trips by start station to get counts and duration summaries;
2. left join the aggregated results to the station table;
3. retain all stations, including those with zero departures;
4. keep the mean duration as missing when there are no observed trips or no observed durations.

Do not replace missing duration measurements with zero.


In [ ]:
# TODO: Build the final station-level table.
# It should aggregate trips by start_station_id and then join back to stations.
# Keep all stations. Use COUNT(trip_id) for recorded departures.
# Use COUNT(duration_minutes) for departures with observed durations.
# Use AVG(duration_minutes) for the mean duration among observed durations only.

query = """
WITH trip_summary AS (
    SELECT
        start_station_id,
        COUNT(trip_id) AS recorded_departure_count,
        COUNT(duration_minutes) AS departures_with_observed_duration,
        AVG(duration_minutes) AS mean_duration_minutes
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
    GROUP BY start_station_id
)
SELECT
    s.station_id,
    s.station_name,
    s.neighborhood,
    COALESCE(ts.recorded_departure_count, 0) AS recorded_departure_count,
    COALESCE(ts.departures_with_observed_duration, 0) AS departures_with_observed_duration,
    ts.mean_duration_minutes
FROM stations s
LEFT JOIN trip_summary ts
    ON s.station_id = ts.start_station_id
ORDER BY s.station_id;
"""

station_summary = pd.read_sql_query(query, conn)
station_summary


## 6. Validate before interpreting

Before making conclusions, check the dataset you have constructed. The final station table should satisfy all of these conditions:

- exactly one row per station;
- unique station IDs in the final table;
- the final row count matches the station table;
- stations with no departures have zero departure counts;
- summed station departure counts equal trips with valid matched start-station IDs;
- matched, unmatched, and missing-key trip counts reconcile with the raw trip count.


In [ ]:
# TODO: Write validation checks for the final station table.
# Check the row count, distinct station IDs, zero departure counts, and reconciliation with the raw trips table.

validations = """
WITH trip_summary AS (
    SELECT
        start_station_id,
        COUNT(trip_id) AS recorded_departure_count,
        COUNT(duration_minutes) AS departures_with_observed_duration,
        AVG(duration_minutes) AS mean_duration_minutes
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
    GROUP BY start_station_id
), station_summary AS (
    SELECT
        s.station_id,
        s.station_name,
        s.neighborhood,
        COALESCE(ts.recorded_departure_count, 0) AS recorded_departure_count,
        COALESCE(ts.departures_with_observed_duration, 0) AS departures_with_observed_duration,
        ts.mean_duration_minutes
    FROM stations s
    LEFT JOIN trip_summary ts
        ON s.station_id = ts.start_station_id
), matched AS (
    SELECT COUNT(*) AS matched_trips
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
), unmatched AS (
    SELECT COUNT(*) AS unmatched_trips
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id NOT IN (SELECT station_id FROM stations)
), missing_key AS (
    SELECT COUNT(*) AS missing_key_trips
    FROM trips
    WHERE start_station_id IS NULL
), raw_total AS (
    SELECT COUNT(*) AS total_trips FROM trips
)
SELECT
    (SELECT COUNT(*) FROM station_summary) AS station_rows,
    (SELECT COUNT(DISTINCT station_id) FROM station_summary) AS distinct_station_ids,
    (SELECT COUNT(*) FROM station_summary WHERE recorded_departure_count = 0) AS stations_with_no_recorded_departures,
    (SELECT SUM(recorded_departure_count) FROM station_summary) AS sum_of_station_departure_counts,
    (SELECT matched_trips FROM matched) AS matched_trips,
    (SELECT unmatched_trips FROM unmatched) AS unmatched_trips,
    (SELECT missing_key_trips FROM missing_key) AS missing_key_trips,
    (SELECT total_trips FROM raw_total) AS total_trips;
"""

pd.read_sql_query(validations, conn)


### Interpretation prompt

Why can the final station totals be smaller than the total number of source trips? Explain the difference between valid matched trips, invalid or unmatched references, and missing keys.


## 7. Analyze in pandas

Now that the station-level table is validated, use it for analysis. Plot the number of recorded departures by station and interpret the pattern.


In [ ]:
station_summary = pd.read_sql_query("""
WITH trip_summary AS (
    SELECT
        start_station_id,
        COUNT(trip_id) AS recorded_departure_count,
        COUNT(duration_minutes) AS departures_with_observed_duration,
        AVG(duration_minutes) AS mean_duration_minutes
    FROM trips
    WHERE start_station_id IS NOT NULL
      AND start_station_id IN (SELECT station_id FROM stations)
    GROUP BY start_station_id
)
SELECT
    s.station_id,
    s.station_name,
    s.neighborhood,
    COALESCE(ts.recorded_departure_count, 0) AS recorded_departure_count,
    COALESCE(ts.departures_with_observed_duration, 0) AS departures_with_observed_duration,
    ts.mean_duration_minutes
FROM stations s
LEFT JOIN trip_summary ts
    ON s.station_id = ts.start_station_id
ORDER BY s.station_id;
""", conn)

station_summary = station_summary.sort_values('recorded_departure_count', ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(station_summary['station_name'], station_summary['recorded_departure_count'], color='#2c7fb8')
ax.set_title('Recorded departures by station')
ax.set_xlabel('Station')
ax.set_ylabel('Recorded departures')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


### Interpret the chart

What pattern stands out?

- Which stations have the highest observed departure counts?
- Which neighborhood or neighborhoods appear most active?
- Do missing durations or unmatched trip records affect your interpretation?
- Does the chart show demand, or only observed departures? Explain the difference.


## 8. Independent practice

Use the same validated station-level table to answer a new question:

**Which neighborhoods have the highest average recorded departures per station during the observation period?**

Your answer should include all of the following:

1. State the intended unit of observation.
2. Use the validated station table, not the raw trips table.
3. Calculate station count, total departures, and mean departures per station for each neighborhood.
4. Include stations with zero departures in the calculations.
5. Create an appropriate visualization.
6. Support your conclusion with numerical evidence.
7. Explain why total departures and average departures per station can produce different rankings.
8. State one limitation of the conclusion.

Do not include the final answer in this notebook. You should work through the logic on your own and justify each choice.


In [ ]:
# TODO: Compute neighborhood-level summary statistics.
# The final output should have one row per neighborhood, with:
# - neighborhood
# - station_count
# - total_departures
# - mean_departures_per_station
# Then create a bar chart for comparison.

# Suggested workflow:
# 1. Start from station_summary.
# 2. Group by neighborhood.
# 3. Sum recorded_departure_count.
# 4. Compute mean recorded_departure_count by neighborhood.
# 5. Plot the mean departures per station by neighborhood.

# neighborhood_summary = ...
# neighborhood_summary
